In [1]:
import torch
import torch.nn as nn
import math
import random

random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


**vocabulary**

In [2]:
special_tokens = ["<pad>", "<bos>", "<eos>"]
base_words = ["one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]

vocab = special_tokens + base_words
stoi = {token: idx for idx, token in enumerate(vocab)}
itos = {idx: token for token, idx in stoi.items()}

In [3]:
PAD_IDX = stoi["<pad>"]
BOS_IDX = stoi["<bos>"]
EOS_IDX = stoi["<eos>"]

vocab_size = len(vocab)
print("Vocab size :", vocab_size)
print(stoi)

Vocab size : 12
{'<pad>': 0, '<bos>': 1, '<eos>': 2, 'one': 3, 'two': 4, 'three': 5, 'four': 6, 'five': 7, 'six': 8, 'seven': 9, 'eight': 10, 'nine': 11}


**Encode / decode helpers**

In [8]:
def encode_tokens(tokens):
    return [stoi[t] for t in tokens]

def decode_tokens(ids):
    words = []
    for idx in ids:
        token = itos[int(idx)]
        if token == "<eos>":
            break
        if token not in ["<pad>", "<bos>"]:
            words.append(token)
    return words

**Create toy dataset**

In [9]:
def generate_sample(min_len=3, max_len=5):
    length = random.randint(min_len, max_len)
    src_words = random.sample(base_words, length)
    tgt_words = list(reversed(src_words))
    return src_words, tgt_words

train_pairs = [generate_sample() for _ in range(1200)]
test_pairs = [generate_sample() for _ in range(200)]

print(train_pairs[:5])

[(['two', 'one', 'six', 'three', 'nine'], ['nine', 'three', 'six', 'one', 'two']), (['three', 'two', 'six'], ['six', 'two', 'three']), (['nine', 'two', 'five', 'four', 'one'], ['one', 'four', 'five', 'two', 'nine']), (['two', 'four', 'nine'], ['nine', 'four', 'two']), (['one', 'four', 'six', 'seven', 'five'], ['five', 'seven', 'six', 'four', 'one'])]


**Batch preparation**

In [10]:
def pad_sequences(sequences, pad_value=PAD_IDX):
    max_len = max(len(seq) for seq in sequences)
    padded = []
    for seq in sequences:
        padded.append(seq + [pad_value] * (max_len - len(seq)))
    
    return torch.tensor(padded, dtype=torch.long)

In [11]:
def make_batch(pairs):
    src_batch = []
    tgt_input_batch = []
    tgt_output_batch = []
    
    for src_words, tgt_words in pairs:
        src_ids = encode_tokens(src_words) + [EOS_IDX]
        tgt_input_ids = [BOS_IDX] + encode_tokens(tgt_words)
        tgt_output_ids = encode_tokens(tgt_words) + [EOS_IDX]
        
        src_batch.append(src_ids)
        tgt_input_batch.append(tgt_input_ids)
        tgt_output_batch.append(tgt_output_ids)

    src_batch = pad_sequences(src_batch)
    tgt_input_batch = pad_sequences(tgt_input_batch)
    tgt_output_batch = pad_sequences(tgt_output_batch)
    
    return src_batch, tgt_input_batch, tgt_output_batch